# Aprendendo a usar o Kaggle

Andreis -----> Ygor

Olá, seja bem vindo ao notebpok de classificadores de ML do Iris Data Science. Nós preparamos esse notebook para que você possa aprender do 0 até o conhecimento básico de uso de ML. Esse material é baseado em dados de competições e nas aulas do Prof. Pascal Yim (E.C. Lille).

Se você nunca utilizou o Kaggle, crie sua conta e clique em "Copy and Edit" neste notebook e começe a programar!

Se você é iniciante, comece aqui pelo começo e siga as instruções. Se você já possui experiência, pode seguir a frente e tentar os desafios!

In [ ]:
import matplotlib.pyplot as matplotlib
import seaborn
import pandas
import numpy
%matplotlib inline

# Função para fazer reshape de listas 1D
def reshape(list1D):
     return numpy.array(list1D).reshape(-1,1)
    
# Função para imprimir nosso modelo de Regressão Logistica
def plot_ours(model):
    x = numpy.linspace(0,1,50)
    y = model.predict(reshape(x))
    matplotlib.figure(figsize=(4,4))
    matplotlib.plot(x, y, color="red")
    matplotlib.suptitle('Our Logistic model')
    matplotlib.xlabel('x')
    matplotlib.ylabel('y')
    matplotlib.show()
    
# Função para imprimir o modelo "padrão" de Regressão Logistica 
def plot_lr():
    logistical = lambda x: numpy.exp(x)/(1+numpy.exp(x))   
    x = numpy.linspace(-10,10,50)
    y = logistical(x)
    matplotlib.figure(figsize=(4,4))
    matplotlib.plot(x, y, color="red")
    matplotlib.suptitle('Logisitc Regression model')
    matplotlib.xlabel('x')
    matplotlib.ylabel('y')
    matplotlib.show()

plot_lr()

Ok, neste começo iremos apresentar o que é um modelo, o que é x, y, labels/test, fit e predict. Lembre-se que toda Inteligência Artificial é um modelo matemático para y = f(x), onde x serão os dados de entrada, y os dados de resposta, e f a nossa função. No caso estamos estudando uma função da forma ```f(x) = exp(x)/(1+numpy.exp(x))```. Nós não iremos entrar na matemática por trás disso mas basta entender que ele é uma curva.

O que faremos a seguir é um "arredondador", ou um classificador de 0's e 1's. Queremos que ele faça 0.00231 -> 0 e 0.7987 -> 1, e assim em diante.

In [ ]:
from sklearn.linear_model import LogisticRegression as lr

# label -> classes de y
# dados de teste -> dados usados pra validação do modelo
# fit = treinar o modelo
# predict = predizer 

x = [0.4, 0.1, 0.7, 0.04, 0.99, 0.00003, 0.863, 0.65, 0.72, 0.34, 0.51, 0.49] # dados de entrada
y = [0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0] # dados de saída

model = lr()
model.fit(reshape(x), y)

Agora que sabemos como fazer o _fit_ do nosso modelo, vamos ver como ele se parece graficamente. E vamos colocar alguns números de entrada para ele tentar adivinhar!

In [ ]:
plot_ours(model)

In [ ]:
test = [0.0004, 0.88884, 0.3445]
result = model.predict(reshape(test))
result

Ok, só que você nunca encontrará na sua vida os dados dessa forma. Eles normalmente estão armazenados em datasets que podem ser acessados por dataframes. A forma mais comum de acessar um dataset é utilizando a biblioteca pandas. Vamos ver esse dataset de dataset de \[0,1\]  que possui mais 10.000 números. (Atenção, por volta de 10% das respostas estão erradas!)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
import random

# Uma função para criar um dataset com 10.000 números entre 0 e 1 classificados como 0 ou 1 (as vezes errado)
def create_dataset():
    x = [random.random() for i in range(10000)]
    classify = lambda i: int(i > 0.5) if random.random() > 0.1 else int(not i > 0.5)
    dataset = pandas.DataFrame(x,columns=['x'])
    dataset['y'] = dataset['x'].apply(classify)
    return dataset 
    
dataset = create_dataset()
dataset # uma tabela com colunas

Veja só como os dados se parecem:

In [ ]:
seaborn.scatterplot(data=dataset,x='x',y='y', alpha=0.01)

Reparem que há entrada erradas, notadas pela parte mais clara do gráfico.

Vamos aplicar novamente o que já sabemos.

In [ ]:
from sklearn.model_selection import train_test_split

# dados de treino -> pro modelo aprender
# dados de teste -> avaliar o modelo

x = dataset['x']
y = dataset['y']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)
# print(x, "-----", y)
model = lr()
model.fit(reshape(x_train), y_train)

result = model.predict(reshape(x_test))
print(f"len(result)={len(result)}")
print(accuracy_score(result, y_test))
# acurácia: qnts previsões corretas ele fez
# métricas são usadas pra saber se o modelo performa bem
conf_matrix = confusion_matrix(result, y_test)
print(conf_matrix) # 2x2

Os números na diagonal principal são as previsões corretas e na diagonal secundária as previsões erradas.

In [ ]:
import numpy as np

# Somar true positives com true negatives e devidir pelo total para achar a acurácia
print((conf_matrix[0][0] + conf_matrix[1][1]) / np.sum(conf_matrix))

### Tentando explicar Matriz de Confusão

Vamos pegar uma entrada pequena **[0.2, 0.4, 0.7, 0.9]**. Sabemos que idealmente suas saídas deveriam ser **[0, 0, 1, 1]**, mas imaginemos que o modelo retorne **[0, 1, 1, 0]**.

https://raw.githubusercontent.com/IRIS-UNICAMP/iniciantes_2022/main/ConfusionMatrix.jpeg

# Mexendo com dados reais (Cancer de Mama)




Vamos predizer cancer! O dataset a seguir é sobre cancêr de mama no estado de Wisconsin. Na coluna do 'diagnosis' podemos ver os dois diagnósticos para cancer de mama: Maligno e Benigno. Vamos tentar adivinhar o diagnóstico baseado apenas nas informações médicas que temos.

In [ ]:
import tree
# Aqui mapeia os dois tipos de diagnósticos no espaço
def plot_cancer_sizes(df):
    matplotlib.figure(figsize=(12,12))
    seaborn.kdeplot(df[df['diagnosis']=='M'].perimeter_worst, df[df['diagnosis']=='M'].area_worst, cmap="Reds",  shade=True, alpha=0.3, shade_lowest=False)
    seaborn.kdeplot(df[df['diagnosis']=='B'].perimeter_worst, df[df['diagnosis']=='B'].area_worst, cmap="Greens", shade=True, alpha=0.3, shade_lowest=False)
    matplotlib.show()

# Faz um plot do perimeter_worst para os dois diagnósticos
def plot_cancer_perimeter(df):
    fig = seaborn.FacetGrid(df, hue="diagnosis", aspect=3)
    fig.map(seaborn.kdeplot, "perimeter_worst", shade=True)
    fig.add_legend()
    matplotlib.show()

# Faz um plot da árvore de decisões
def plot_tree(model,x_train):
    matplotlib.figure(figsize=(15,15))
    tree.plot_tree(model, feature_names=x_train.columns, class_names=['benigno','maligno'], fontsize=14, filled=True)
    matplotlib.show()

# Faz um plot das importâncias para um Random Forest
def plot_importances(model,df):
    importances = model.feature_importances_
    indices = numpy.argsort(importances)
    matplotlib.figure(figsize=(12,8))
    matplotlib.barh(range(len(indices)), importances[indices], color='b', align='center')
    matplotlib.yticks(range(len(indices)), df.columns[indices])
    matplotlib.suptitle('Importância das características')
    matplotlib.show()

cancer = pandas.read_csv('../input/breast-cancer-wisconsin-data/data.csv')
cancer
# NaN = null, 0, vazio, not a number

Primeiro passo: Tire a coluna inútil no final!

In [ ]:
cancer = cancer.drop(columns=["Unnamed: 32"])
cancer

Agora, vamos visualizar como o tamanho do perímetro e tamanho da área se comportam para o tipo maligno e benigno

In [ ]:
plot_cancer_sizes(cancer)


Ok, faça x com todas as colunas menos o diagnóstico, e y como diagnóstico! Lembre de separar os dados e aplicar a Regressão Logística

In [ ]:
import pandas as pd

x = cancer.drop(columns=["diagnosis"])
y = pd.DataFrame(cancer["diagnosis"]) # tipos series não tem o método replace

dict_val = {"M": 0, "B": 1}

y = y.replace({"diagnosis": dict_val})

x_train, x_test, y_train, y_test = train_test_split(x, y["diagnosis"].ravel(), test_size=0.2, random_state=0)

model = lr()
model.fit(x_train, y_train) # treinar

result = model.predict(x_test)
print(accuracy_score(result, y_test))
print(confusion_matrix(result, y_test))

# Maligno e Benigno
# [0,          1]
# [1,          0]

#   0          1

# acurácia: número de previsões correta de todas as previsões feitas
# one hot encoding

A acurácia foi ruim, como você pode ver. O que aconteceu? Será que tem alguma coluna que está fazendo nossos dados tendenciosos? (dica: tem sim)

In [ ]:
## Vamos contar quantos M e B há no dataset
print("M = " + str(cancer[cancer["diagnosis"] == "M"].shape[0]))
print("B = " + str(cancer[cancer["diagnosis"] == "B"].shape[0]))

Então a chance de ser M é 212/(212+357) ---> 0.37258347978910367  
A chande de ser B do 357/(212+357) ---> 0,62741652

In [ ]:
from sklearn.preprocessing import QuantileTransformer
# remover a coluna id
try:
    x = x.drop(["id"], axis=1)
except:
    pass
# cuidado ao rodar a célula mais de uma vez por causa da remoção da coluna id
# o y é o definido nas celulas anteriores

x_train, x_test, y_train, y_test = train_test_split(x, y["diagnosis"].ravel(), test_size=0.2, random_state=0)

quantile_transformer = QuantileTransformer(random_state=0)
x_train_trans = quantile_transformer.fit_transform(x_train)
x_test_trans = quantile_transformer.transform(x_test)

model = lr()
model.fit(x_train_trans, y_train) #treinar

result = model.predict(x_test_trans)
print(accuracy_score(result, y_test))
print(confusion_matrix(result, y_test))

Lembre-se sempre da aplicação de IA: classificar coisas! Imagine que eu sou um médico com um paciente com essas condições. O câncer é maligno ou benigno?

In [ ]:
nosso_cancer = list({
 'radius_mean': 17.99,
 'texture_mean': 10.38,
 'perimeter_mean': 122.8,
 'area_mean': 1001.0,
 'smoothness_mean': 0.1184,
 'compactness_mean': 0.2776,
 'concavity_mean': 0.3001,
 'concave points_mean': 0.1471,
 'symmetry_mean': 0.2419,
 'fractal_dimension_mean': 0.07871,
 'radius_se': 1.095,
 'texture_se': 0.9053,
 'perimeter_se': 8.589,
 'area_se': 153.4,
 'smoothness_se': 0.006399,
 'compactness_se': 0.04904,
 'concavity_se': 0.05372999999999999,
 'concave points_se': 0.01587,
 'symmetry_se': 0.03003,
 'fractal_dimension_se': 0.006193,
 'radius_worst': 25.38,
 'texture_worst': 17.33,
 'perimeter_worst': 184.6,
 'area_worst': 2019.0,
 'smoothness_worst': 0.1622,
 'compactness_worst': 0.6656,
 'concavity_worst': 0.7119,
 'concave points_worst': 0.2654,
 'symmetry_worst': 0.4601,
 'fractal_dimension_worst': 0.1189
}.values())

model.predict(reshape(nosso_cancer).transpose())

# dict_val = {"M": 0, "B": 1}

Ok, cansamos de usar Logistic Regression. Queremos modelos diferentes para fazer aprendizado de máquina! Vamos tentar utilizar uma árvore de decisões, é o mesmo "modelo" que o Akinator funcionava!

In [ ]:
from sklearn import tree

clf = tree.DecisionTreeClassifier()
clf = clf.fit(x_train, y_train)
result = clf.predict(x_test)
print(accuracy_score(result, y_test))
print(confusion_matrix(result, y_test))

Vamos olhar com a árvore de decisão se parece visualmente!

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(21,21))
tree.plot_tree(clf, filled=True, class_names=["M", "B"])

Ok, agora vamos ver outro tipo de modelo chamado RandomForestClassifier. Veja que o modelo de aplicação no código é o mesmo!

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier()
clf = clf.fit(x_train, y_train)
result = clf.predict(x_test)
print(accuracy_score(result, y_test))
print(confusion_matrix(result, y_test))

E agora vamos aprender um pouco sobre classificação, como precisão, recall, f1-score, e **LOSS**

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, result))

https://imasters.com.br/desenvolvimento/machine-learning-metricas-para-modelos-de-classificacao  
https://static.imasters.com.br/wp-content/uploads/2019/04/11135729/vEYRtVQ.png

Acurácia = número predições certas / número de predições  
Precisão = a ideia é olhar só o número de classes positivas que são realmente positivas  
Precision = TruePositives / (TruePositives + FalsePositives)
0 precisao ou precisão 1

Recall = Recall = TruePositives / (TruePositives + FalseNegatives)


### Faltou explicar o que é overfitting e underfitting.


# IRIS

Iris, o desafio que deu nome ao nome do nosso grupo, é um dataset do tipo de flor Iris (e não o olho!). Nele, há três tipos de espécies, com as informações sobre sépalas e pétalas.

In [ ]:
def plot_iris(df):
    seaborn.pairplot(df, hue="Species")
    plt.show()

#### (1) Leia o dataset para dataframe usando a biblioteca pandas
#### (2) Mostre o dataframe

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
# o caminho do csv é "../input/iris/Iris.csv"

import pandas as pd

path = "../input/iris/Iris.csv"
df = pd.read_csv(path)
df

#### (3) Remova a coluna "Id" do dataframe

#### (4) Chame a função plot_iris() com o dataframe lido na célula anterior como argumento

#### (5) Faça um análise mental rápida dos gráficos mostrados

In [ ]:
import seaborn
import matplotlib
import matplotlib.pyplot as plt

df_new = df.drop(columns=["Id"])
plot_iris(df_new)

In [ ]:
import plotly
import plotly.express as px

def plot_3d(df_new):
    fig = px.scatter_3d(df_new,
                        x = df_new.columns[0],
                        y = df_new.columns[1],
                        z = df_new.columns[3],
                        size = df_new.columns[2],
                        color = df_new.columns[4],
                        opacity = 0.7)

    fig.update_layout(margin = dict(l=0, r=0, b=0, t=0))
    fig.show()

plot_3d(df_new)

#### Vamos aplicar nossa regressão logística.

In [ ]:
### Dicas: divida o dataframe em x e y, depois chame a função train_test_split do sklearn para dividir
### entre dados de treino e teste.
# https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
# Mostre a acurácia, precisão, matriz de confusão e recall para os dados de teste

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

# dados normalizados
x = preprocessing.normalize(df_new.drop(columns=["Species"]))
y = pd.DataFrame(df_new["Species"])

y = y.apply(LabelEncoder().fit_transform) # transformar classes de iris para números

x_train, x_test, y_train, y_test = train_test_split(x, pd.Series(y["Species"]).ravel(), test_size=0.2, random_state=0)

model = LogisticRegression()
model.fit(x_train, y_train) #treinar

result = model.predict(x_test)
print(accuracy_score(result, y_test))
print(confusion_matrix(result, y_test))

A acurácia não foi boa. Foi péssima na verdade.
#### Usando uma árvore de decisão

In [ ]:
from sklearn import tree

clf = tree.DecisionTreeClassifier()
clf = clf.fit(x_train, y_train)
result = clf.predict(x_test)
print(accuracy_score(result, y_test))
print(confusion_matrix(result, y_test))

# Desafio 1: Titanic

O seu objetivo é descobrir, dado as informações de um passageiro no navio Titanic, se ele sobreviveu ou não o acidente (coluna survived). Neste caso, você perceberá que algumas colunas podem deixar seu aprendizado pior. Outro ponto importante é que há varias informações faltantes (NaN).

**Objetivos**
- Você consegue preencher as informações faltantes de alguma forma? Como?
- Tente ultrapassar 80% de precisão.




Passos:

1) Olhe para o conjunto de dados e tente entendê-lo
https://www.kaggle.com/competitions/titanic/data

2) Leia o dataset de treino e teste
path_test = "../input/titanic/test.csv"
path_train = "../input/titanic/train.csv"

3) Análise quais colunas são importantes para o modelo e remova as inúteis

4) Análise campos com NaN, 0 ou nulo e decida o que fazer com eles

5) Monte o modelo e mostre a acurácia, matriz de confusão (recall e precisão também se quiser)

# Desafio 2: IMDB

Ok, o seu desafio agora é utilizar Machine Learning para fazer uma Análise de Sentimentos nas reviews de filmes do IMDB. Neste dataset você possui 50.000 reviews de filmes classificadas como "positiva" e "negativa". Neste dataset, seu modelo pode demorar bastante (coisa de 5 minutos para cima). O quê você precisa fazer:

- Limpar as reviews (use a função clear_sentence para cada string do dataset)
- Separe o treinamento e teste.
- Vetorizar as palavras (pode usar o HashingVectorizer()). Procure no google como aplicar isso.
- Coloque em um Machine Learning.

**Objetivos**
- Qual é o melhor tipo de modelo de ML para esse NLP? (Dica: pense em modelos que trabalham com vetores)
- Ultrapasse 90% de precisão neste dataset demorando menos de 1 minuto para rodar (utilize time.time() para pegar os tempos.




In [ ]:
import string
import time

from sklearn.feature_extraction.text import HashingVectorizer

def clear_sentence(sentence):
    sentence = sentence.replace('<br />', ' ')
    sentence = sentence.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
    sentence = sentence.lower()
    return sentence

imdb = pandas.read_csv("../input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
imdb

# Desafio 3: Diabetes
O dataset a seguir possui dados sobre a incidência de diabetes na população do povo Pima. Seu desafio é descobrir a coluna outcome baseado nos outros dados. Seu desafio é ultrapassar 82% de precisão.

In [ ]:
diabetes = pandas.read_csv("/kaggle/input/pima-indians-diabetes-database/diabetes.csv")
diabetes

# Desafio 4: MNIST - Digit Recognizer
Dataset com dígitos escritos à mão e seus respectivos valores. Cada linha dos datasets (tanto de treino quanto de teste) está estruturada da seguinte forma:

| Digito representado | pixel 1x1 | ... | pixel 28x28 |
|:-----------------:|:---------:|:---:|:----------:|
|5|0|...|0|

Como temos uma imagem 28x28 temos 784 valores de pixel por coluna, todos valores binários:

In [ ]:
def plt_digit_from_row(row):
    label, image = mnist_train.values[row,0], mnist_train.values[row,1:]
    matplotlib.imshow(image.reshape(28,28), cmap='hot')
    matplotlib.title("Label: %s"%label)
    matplotlib.show()

mnist_train = pandas.read_csv("../input/mnist-in-csv/mnist_train.csv")
mnist_test = pandas.read_csv("../input/mnist-in-csv/mnist_test.csv")
mnist_train.head()

Conseguimos ver como é a imagem redimensionando o a matriz `1x784` para uma `28x28`:

In [ ]:
plt_digit_from_row(0)

In [ ]:
mnist_train_labels, mnist_train_values = mnist_train.values[:,0], mnist_train.values[:,1:]
mnist_test_labels, mnist_test_values = mnist_test.values[:,0], mnist_test.values[:,1:]

Você vai precisar fazer o escalamento das imagens para poder 

In [ ]:
model = LogisticRegression()

model.fit(mnist_train_values, mnist_train_labels)


prediction = model.predict(mnist_test_values)

print(classification_report(prediction, mnist_test_labels))

# Desafio 5: Fashion MNIST
Dataset com desenhos de tipos de roupa classificadas com labels
Cada linha dos datasets (tanto de treino quanto de teste) está estruturada da seguinte forma:


| Label de cada roupa | pixel 1x1 | ... | pixel 28x28 |
|:-----------------:|:---------:|:---:|:----------:|
|5|0|...|0|

Como temos uma imagem 28x28 temos 784 valores de pixel por coluna, todos valores binários:

In [ ]:
def plt_clothes_from_row(row):
    label, image = fashion_mnist_train.values[row,0], fashion_mnist_train.values[row,1:]
    matplotlib.imshow(image.reshape(28,28), cmap='gray')
    matplotlib.title("Label: %s"%label)
    matplotlib.show()
    
fashion_mnist_train, fashion_mnist_test = pandas.read_csv("../input/fashionmnist/fashion-mnist_train.csv"), pandas.read_csv("../input/fashionmnist/fashion-mnist_test.csv")
fashion_mnist_train.head()

In [ ]:
plt_clothes_from_row(0)

In [ ]:
fashion_mnist_train_labels, fashion_mnist_train_values = fashion_mnist_train.values[:,0], fashion_mnist_train.values[:,1:]
fashion_mnist_test_labels, fashion_mnist_test_values = fashion_mnist_test.values[:,0], fashion_mnist_test.values[:,1:]

In [ ]:
model = LogisticRegression()

model.fit(fashion_mnist_train_values, fashion_mnist_train_labels)

prediction = model.predict(fashion_mnist_test_values)

print(classification_report(prediction, mnist_test_labels))

# Desafio Final: Predizer eleições com Tweets

Dessa vez nem preparamos o dataset para você. Utilizando o dataset das eleições australianas, você consegue predizer que regiões da Austrália apoiam qual partido? Tente treinar seu modelo em algum dataset classificado com positivo e negativo e então faça o .fit() no dataset das eleições. Você pode utilizar a função a seguir para plotar seus dados.

In [ ]:
from mpl_toolkits.basemap import Basemap

# Precisa ter as colunas 'lat' e 'long'. Retorna o mesmo dataframe com apenas os tweets na região da austrália.
def pegar_tweets_na_australia(dataframe):
    bot_lat, top_lat, left_lon, right_lon = -44,-10,109,156
    top = dataframe.lat <= top_lat
    bot = dataframe.lat >= bot_lat
    left = dataframe.long >= left_lon
    right = dataframe.long <= right_lon
    index = top&bot&left&right 
    return dataframe[index]

# Passe seu dataframe com os dados que você quer plotar e uma legenda (como string).
def plotar_mapa(dataframe,legenda):
    Australia_map = Basemap(llcrnrlat=-44,urcrnrlat=-10,llcrnrlon=109,urcrnrlon=156)
    matplotlib.figure(figsize=(12,10))
    Australia_map.bluemarble(alpha=0.9)
    seaborn.scatterplot(x='long', y='lat', data=dataframe, alpha=1, s=200, label=legenda)
    matplotlib.show()